## XGBoost Pairwise Feature Validation Entry

This notebook is the handoff entry for feature experiments. It uses rolling k-fold validation from `code/utils/validation.py` while leaving the formal holdout config in `code/config.py` unchanged.

- Split source: `build_validation_plan(raw_df, config).validation_splits()`
- Experiment mode: notebook-local `rolling_kfold`
- Device: from `config['model_params']['xgb_rank_pairwise']['device']`
- Output: `output/xgb_pairwise_feature_validation_rolling_kfold.json`


In [1]:
import copy
import json
import os
import random
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'code').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
sys.modules.pop('code', None)

import numpy as np
import pandas as pd
import xgboost as xgb

from code.config import config as base_config
from code.features.baseline import preprocess_stock_data_samples
from code.features.windows import feature_num_for_window
from code.models.xgboost.loss import topk_return_metrics, xgb_rank_ic_metric, xgb_rank_return_metrics
from code.models.xgboost.model import TqdmTrainingCallback, choose_best_iteration, make_dmatrix, target_range
from code.utils.log import get_logger, log_json
from code.utils.runtime_split import load_market_data
from code.utils.validation import build_validation_plan

config = copy.deepcopy(base_config)
config['validation'] = {**config['validation'], 'mode': 'rolling_kfold'}

MODEL_NAME = 'xgb_rank_pairwise'
EXPERIMENTS = (
    {'input_window': 1, 'feature_type': 'window'},
    {'input_window': 2, 'feature_type': 'window'},
    {'input_window': 4, 'feature_type': 'window'},
    {'input_window': 8, 'feature_type': 'window'},
    {'input_window': 12, 'feature_type': 'window'},
)
OUTPUT_PATH = Path('output/xgb_pairwise_window_factor_validation_rolling_kfold.json')
LOG_PATH = Path('output/xgb_pairwise_window_factor_validation_rolling_kfold.log')
logger = get_logger('xgb_pairwise_window_factor_validation_rolling_kfold', LOG_PATH)


/data/zzj/BDC2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed):
    # 固定实验随机性。
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


def model_params():
    # 复用正式 pairwise 参数，只移除配置绑定字段。
    params = {
        key: value
        for key, value in config['model_params'][MODEL_NAME].items()
        if key not in {'input_window', 'feature_type'}
    }
    params['disable_default_eval_metric'] = 1
    return params


def split_meta(fold):
    # 记录连续时间块，完全来自 utils.validation.ValidationFold。
    return {
        'fold': fold.name,
        'train_start': str(fold.train_start.date()),
        'train_end': str(fold.train_end.date()),
        'validation_start': str(fold.validation_start.date()),
        'validation_end': str(fold.validation_end.date()),
    }


def train_eval_fold(input_window, feature_type, feature_num, features, fold, train_df, val_df):
    # 单个特征配置在单个时间 fold 上训练评估。
    dtrain, train_groups = make_dmatrix(train_df, features)
    dval, val_groups = make_dmatrix(val_df, features)
    evals_result = {}
    started = time.time()
    num_boost_round = int(config['num_boost_round'])
    booster = xgb.train(
        params=model_params(),
        dtrain=dtrain,
        num_boost_round=num_boost_round,
        evals=[(dtrain, 'train'), (dval, 'validation')],
        custom_metric=xgb_rank_return_metrics,
        maximize=True,
        evals_result=evals_result,
        verbose_eval=False,
        callbacks=[TqdmTrainingCallback(num_boost_round, f'w{input_window} {feature_type} {fold.name}')],
    )
    selection = choose_best_iteration(
        [float(value) for value in evals_result['validation']['top5_excess_return']],
        'validation_top5_excess_return',
    )
    best_iteration = int(selection['best_iteration'])
    val_pred = booster.predict(dval, iteration_range=(0, best_iteration + 1))
    val_top5 = topk_return_metrics(val_pred, dval.get_label(), val_groups, top_k=5)
    val_top10 = topk_return_metrics(val_pred, dval.get_label(), val_groups, top_k=10)
    return {
        **split_meta(fold),
        'model': MODEL_NAME,
        'input_window': input_window,
        'feature_type': feature_type,
        'feature_num': feature_num,
        'features': len(features),
        'rounds': num_boost_round,
        'best_iteration': best_iteration,
        'best_selection_metric': selection['metric'],
        'best_validation_top5_return': float(evals_result['validation']['top5_return'][best_iteration]),
        'best_validation_top5_excess_return': float(selection['score']),
        'best_validation_top5_precision': float(evals_result['validation']['top5_precision'][best_iteration]),
        'validation_rank_ic': float(xgb_rank_ic_metric(val_pred, dval)[1]),
        'validation_universe_return': float(val_top5['benchmark_top5_return_avg']),
        'validation_top5_return': float(val_top5['pred_top5_return_avg']),
        'validation_top5_excess_return': float(val_top5['pred_top5_excess_return_avg']),
        'validation_top5_excess_return_std': float(val_top5['pred_top5_excess_return_std']),
        'validation_top5_excess_return_min': float(val_top5['pred_top5_excess_return_min']),
        'validation_top5_excess_positive_rate': float(val_top5['pred_top5_excess_positive_rate']),
        'validation_top5_precision': float(val_top5['pred_top5_precision_avg']),
        'validation_top10_return': float(val_top10['pred_top10_return_avg']),
        'validation_top10_excess_return': float(val_top10['pred_top10_excess_return_avg']),
        'validation_top10_excess_return_std': float(val_top10['pred_top10_excess_return_std']),
        'validation_top10_excess_return_min': float(val_top10['pred_top10_excess_return_min']),
        'validation_top10_excess_positive_rate': float(val_top10['pred_top10_excess_positive_rate']),
        'validation_top10_precision': float(val_top10['pred_top10_precision_avg']),
        'validation_top5_by_week': val_top5['pred_top5_group_returns'],
        'validation_top5_excess_by_week': val_top5['pred_top5_excess_group_returns'],
        'validation_top10_by_week': val_top10['pred_top10_group_returns'],
        'validation_top10_excess_by_week': val_top10['pred_top10_excess_group_returns'],
        'train_groups': len(train_groups),
        'validation_groups': len(val_groups),
        'train_rows': int(dtrain.num_row()),
        'validation_rows': int(dval.num_row()),
        'seconds': time.time() - started,
    }


def summarize(rows):
    # 按 rolling fold 聚合，fold_count 用于确认 std 来自多折验证。
    df = pd.DataFrame(rows)
    metric_cols = [
        'best_validation_top5_return',
        'best_validation_top5_excess_return',
        'best_validation_top5_precision',
        'validation_rank_ic',
        'validation_universe_return',
        'validation_top5_return',
        'validation_top5_excess_return',
        'validation_top5_excess_return_std',
        'validation_top5_excess_return_min',
        'validation_top5_excess_positive_rate',
        'validation_top5_precision',
        'validation_top10_return',
        'validation_top10_excess_return',
        'validation_top10_excess_return_std',
        'validation_top10_excess_return_min',
        'validation_top10_excess_positive_rate',
        'validation_top10_precision',
    ]
    grouped = df.groupby(['input_window', 'feature_type', 'feature_num', 'features'], sort=False)
    summary = grouped[metric_cols].agg(['mean', 'std', 'min', 'max'])
    summary.columns = ['_'.join(col).rstrip('_') for col in summary.columns]
    summary['fold_count'] = grouped.size()
    summary = summary.reset_index()
    front_cols = ['input_window', 'feature_type', 'feature_num', 'features', 'fold_count']
    return summary[front_cols + [col for col in summary.columns if col not in front_cols]]


def leaderboard(summary):
    # 只展示最关键的模型选择列，完整 summary 仍写入 JSON。
    columns = [
        'input_window',
        'feature_type',
        'features',
        'fold_count',
        'validation_top5_excess_return_mean',
        'validation_top5_excess_return_std',
        'validation_top5_excess_return_min',
        'validation_top5_excess_positive_rate_mean',
        'validation_top5_precision_mean',
        'validation_rank_ic_mean',
        'validation_top5_return_mean',
        'validation_universe_return_mean',
    ]
    available = [col for col in columns if col in summary.columns]
    return summary.sort_values('validation_top5_excess_return_mean', ascending=False)[available].reset_index(drop=True)


In [3]:
set_seed(int(config['seed']))
started = time.time()
raw_df = load_market_data(config)
stock_ids = sorted(raw_df['股票代码'].unique())
stockid2idx = {sid: idx for idx, sid in enumerate(stock_ids)}
plan = build_validation_plan(raw_df, config)
folds = plan.validation_splits()
log_json(logger, 'validation_config', config['validation'])
log_json(logger, 'folds', [split_meta(fold) for fold in folds])
logger.info('data rows=%s stocks=%s date=%s..%s', len(raw_df), len(stock_ids), raw_df['日期'].min().date(), raw_df['日期'].max().date())

rows = []
datasets = {}
for experiment in EXPERIMENTS:
    input_window = int(experiment['input_window'])
    feature_type = experiment['feature_type']
    feature_num = feature_num_for_window(input_window, feature_type)
    dataset_key = f'{input_window}:{feature_type}'
    datasets[dataset_key] = {'input_window': input_window, 'feature_type': feature_type, 'feature_num': feature_num, 'folds': []}
    for fold in folds:
        logger.info('prepare input_window=%s feature_type=%s fold=%s', input_window, feature_type, fold.name)
        train_samples = plan.get_train_samples(fold, input_window)
        val_samples = plan.get_validation_samples(fold, input_window)
        train_df, features = preprocess_stock_data_samples(train_samples, feature_num, stockid2idx)
        val_df, _ = preprocess_stock_data_samples(val_samples, feature_num, stockid2idx)
        train_df = train_df.dropna(subset=['label']).sort_values(['日期', '股票代码']).reset_index(drop=True)
        val_df = val_df.dropna(subset=['label']).sort_values(['日期', '股票代码']).reset_index(drop=True)
        fold_data = {
            **split_meta(fold),
            'features': len(features),
            'train_rows': len(train_df),
            'validation_rows': len(val_df),
            'train_target_range': target_range(train_df),
            'validation_target_range': target_range(val_df),
        }
        datasets[dataset_key]['folds'].append(fold_data)
        logger.info('train input_window=%s feature_type=%s fold=%s features=%s', input_window, feature_type, fold.name, len(features))
        rows.append(train_eval_fold(input_window, feature_type, feature_num, features, fold, train_df, val_df))

summary = summarize(rows)
payload = {
    'experiment': 'xgb_rank_pairwise_window_factor_validation',
    'split_source': 'code.utils.validation.build_validation_plan(...).validation_splits()',
    'seed': int(config['seed']),
    'num_boost_round': int(config['num_boost_round']),
    'validation_config': config['validation'],
    'folds': [split_meta(fold) for fold in folds],
    'experiments': EXPERIMENTS,
    'datasets': datasets,
    'rows': rows,
    'summary': summary.to_dict('records'),
    'elapsed_seconds': time.time() - started,
}
OUTPUT_PATH.parent.mkdir(exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
logger.info('saved=%s elapsed=%.2fs', OUTPUT_PATH, payload['elapsed_seconds'])
leaderboard(summary)


2026-07-06 17:22:26 | INFO | xgb_pairwise_window_factor_validation_rolling_kfold | validation_config={"mode": "rolling_kfold", "num_test_weeks": 4, "num_validation_weeks": 8, "rolling_folds": 4, "rolling_gap_weeks": 0, "rolling_min_train_weeks": 120, "rolling_validation_weeks": 4}


2026-07-06 17:22:26 | INFO | xgb_pairwise_window_factor_validation_rolling_kfold | folds=[{"fold": "rolling_1", "train_end": "2026-01-26", "train_start": "2023-01-09", "validation_end": "2026-03-09", "validation_start": "2026-01-26"}, {"fold": "rolling_2", "train_end": "2026-03-09", "train_start": "2023-01-09", "validation_end": "2026-04-13", "validation_start": "2026-03-09"}, {"fold": "rolling_3", "train_end": "2026-04-13", "train_start": "2023-01-09", "validation_end": "2026-05-25", "validation_start": "2026-04-13"}, {"fold": "rolling_4", "train_end": "2026-05-25", "train_start": "2023-01-09", "validation_end": "2026-06-29", "validation_start": "2026-05-25"}]


2026-07-06 17:22:26 | INFO | xgb_pairwise_window_factor_validation_rolling_kfold | data rows=250233 stocks=300 date=2023-01-03..2026-06-26


2026-07-06 17:22:26 | INFO | xgb_pairwise_window_factor_validation_rolling_kfold | prepare input_window=1 feature_type=window fold=rolling_1


2026-07-06 17:22:43 | INFO | xgb_pairwise_window_factor_validation_rolling_kfold | train input_window=1 feature_type=window fold=rolling_1 features=69


...


w12 window rolling_4: 100%|█| 200/200 [00:20<00:00,  8.50round/s, val_excess=0.


w12 window rolling_4: 100%|█| 200/200 [00:20<00:00,  8.50round/s, val_excess=0.


w12 window rolling_4: 100%|█| 200/200 [00:20<00:00,  9.58round/s, val_excess=0.

2026-07-06 17:36:13 | INFO | xgb_pairwise_window_factor_validation_rolling_kfold | saved=output/xgb_pairwise_window_factor_validation_rolling_kfold.json elapsed=827.97s


,input_window,feature_type,features,fold_count,validation_top5_excess_return_mean,validation_top5_excess_return_std,validation_top5_excess_return_min,validation_top5_excess_positive_rate_mean,validation_top5_precision_mean,validation_rank_ic_mean,validation_top5_return_mean,validation_universe_return_mean
0,12,window,69,4,0.038844,0.009948,0.028770,0.8125,0.0875,0.084766,0.033477,-0.005367
1,2,window,69,4,0.037731,0.015105,0.019274,0.7500,0.1000,0.044837,0.032343,-0.005388
2,1,window,69,4,0.034167,0.015719,0.019352,0.9375,0.0750,0.042551,0.028778,-0.005388
3,4,window,69,4,0.032193,0.016603,0.010871,0.9375,0.0625,0.048188,0.026804,-0.005388
4,8,window,69,4,0.021752,0.014358,0.004610,0.7500,0.0375,0.089284,0.016360,-0.005392


In [4]:
captured = json.loads(OUTPUT_PATH.read_text(encoding='utf-8'))
leaderboard(pd.DataFrame(captured['summary']))


,input_window,feature_type,features,fold_count,validation_top5_excess_return_mean,validation_top5_excess_return_std,validation_top5_excess_return_min,validation_top5_excess_positive_rate_mean,validation_top5_precision_mean,validation_rank_ic_mean,validation_top5_return_mean,validation_universe_return_mean
0,12,window,69,4,0.038844,0.009948,0.028770,0.8125,0.0875,0.084766,0.033477,-0.005367
1,2,window,69,4,0.037731,0.015105,0.019274,0.7500,0.1000,0.044837,0.032343,-0.005388
2,1,window,69,4,0.034167,0.015719,0.019352,0.9375,0.0750,0.042551,0.028778,-0.005388
3,4,window,69,4,0.032193,0.016603,0.010871,0.9375,0.0625,0.048188,0.026804,-0.005388
4,8,window,69,4,0.021752,0.014358,0.004610,0.7500,0.0375,0.089284,0.016360,-0.005392
